# 电商用户行为分析项目：数据清洗

本Notebook用于读取淘宝用户行为数据集 UserBehavior.csv，完成字段命名、数据规模查看、缺失值检查、重复值处理、时间字段转换和基础字段扩展。


In [12]:
# 导入数据处理所需依赖
import pandas as pd
import numpy as np


In [13]:
# 读取原始用户行为数据，并补充字段名称
df = pd.read_csv(
    "../data/UserBehavior.csv",
    header=None,
    names=["user_id", "item_id", "category_id", "behavior_type", "timestamp"]
)

df.head()


,user_id,item_id,category_id,behavior_type,timestamp
0,1,2268318,2520377,pv,1511544070
1,1,2333346,2520771,pv,1511561733
2,1,2576651,149192,pv,1511572885
3,1,3830808,4181361,pv,1511593493
4,1,4365585,2520377,pv,1511596146


In [14]:
# 查看数据规模、字段信息和行为类型分布
print("数据规模：", df.shape)
df.info()
display(df.head())
display(df["behavior_type"].value_counts())


数据规模： (100150807, 5)
<class 'pandas.DataFrame'>
RangeIndex: 100150807 entries, 0 to 100150806
Data columns (total 5 columns):
 #   Column         Dtype
---  ------         -----
 0   user_id        int64
 1   item_id        int64
 2   category_id    int64
 3   behavior_type  str  
 4   timestamp      int64
dtypes: int64(4), str(1)
memory usage: 3.9 GB


,user_id,item_id,category_id,behavior_type,timestamp
0,1,2268318,2520377,pv,1511544070
1,1,2333346,2520771,pv,1511561733
2,1,2576651,149192,pv,1511572885
3,1,3830808,4181361,pv,1511593493
4,1,4365585,2520377,pv,1511596146


behavior_type
pv      89716264
cart     5530446
fav      2888258
buy      2015839
Name: count, dtype: int64

In [15]:
# 当数据量过大时进行随机抽样，便于本地分析和复现
if len(df) > 5000000:
    df = df.sample(5000000, random_state=42)

print("抽样后数据规模：", df.shape)


抽样后数据规模： (5000000, 5)


In [16]:
# 检查各字段缺失值情况
missing_values = df.isnull().sum()
print("缺失值统计：")
print(missing_values)


缺失值统计：
user_id          0
item_id          0
category_id      0
behavior_type    0
timestamp        0
dtype: int64


In [17]:
# 删除重复记录，并查看处理后的数据规模
df.drop_duplicates(inplace=True)
print("去重后数据规模：", df.shape)


去重后数据规模： (5000000, 5)


In [18]:
# 将时间戳转换为日期时间，并扩展日期、小时、星期字段
df["time"] = pd.to_datetime(df["timestamp"], unit="s")
df["date"] = df["time"].dt.date
df["hour"] = df["time"].dt.hour
df["weekday"] = df["time"].dt.day_name()

display(df.head())


,user_id,item_id,category_id,behavior_type,timestamp,time,date,hour,weekday
43314727,237858,1062152,1464116,pv,1512212331,2017-12-02 10:58:51,2017-12-02,10,Saturday
88035905,44200,3916487,4135836,pv,1511931558,2017-11-29 04:59:18,2017-11-29,4,Wednesday
38200464,920479,2816757,154040,pv,1512311001,2017-12-03 14:23:21,2017-12-03,14,Sunday
83578911,239462,2698818,154040,pv,1511783539,2017-11-27 11:52:19,2017-11-27,11,Monday
78996463,947273,3991295,1029459,pv,1512273837,2017-12-03 04:03:57,2017-12-03,4,Sunday


In [19]:
# 保存清洗后的数据，供后续EDA和看板分析使用
df.to_csv("../data/clean_user_behavior.csv", index=False)
print("清洗结果已保存至 ../data/clean_user_behavior.csv")


清洗结果已保存至 ../data/clean_user_behavior.csv


In [20]:
# 输出清洗总结信息，方便快速复盘数据情况
print("清洗后数据量：", len(df))
print("字段数量：", df.shape[1])
print("行为类型分布：")
print(df["behavior_type"].value_counts())
print("时间范围：", df["time"].min(), "至", df["time"].max())


清洗后数据量： 5000000
字段数量： 9
行为类型分布：
behavior_type
pv      4479844
cart     275690
fav      143845
buy      100621
Name: count, dtype: int64
时间范围： 1920-10-09 19:25:14 至 2036-10-20 18:19:17
